# `main.ipynb` - Pipeline GeoKW (Índice de Oportunidad de Inversión, Sevilla)

**Autor:** Francisco Máximo Ortega Calvo
**Área:** Data & BI

Punto de entrada del proyecto. Sistema modular de Ingeniería de Datos Espaciales que automatiza la extracción de los 11 distritos de Sevilla y de los puntos de recarga de vehículo eléctrico, los cruza con variables socioeconómicas oficiales, calcula el **Índice de Oportunidad de Inversión (IOI)** y lo representa en un visor cartográfico interactivo (`output/mapa_sevilla.html`).

El pipeline está organizado como un conjunto de **notebooks-módulo** dentro de `geokw_etl/` (uno por responsabilidad), encadenados con `%run` desde este notebook - igual que en el proyecto GeoStat.

| Notebook-módulo (`geokw_etl/`) | Responsabilidad |
|---|---|
| `config.ipynb` | Parámetros, rutas y tabla maestra socioeconómica (data seeding) |
| `logging_config.ipynb` | Configuración centralizada del logging |
| `extract_distritos.ipynb` | Carga y validación del GeoJSON de distritos (IDE Sevilla) |
| `extract_socioeconomico.ipynb` | Tabla maestra de población y renta por distrito |
| `extract_cargadores.ipynb` | Extracción resiliente de puntos de recarga (Open Charge Map + fallback) |
| `transform_geoprocessing.ipynb` | Spatial join puntos↔distritos y agregación de potencia |
| `transform_kpis.ipynb` | Densidad energética, normalización min-max, IOI |
| `visualize_mapa.ipynb` | Visor cartográfico Folium (los 5 requisitos UX/UI) |
| `pipeline.ipynb` | Orquestador `ejecutar_pipeline()` |

| Sección de este notebook | Contenido |
|---|---|
| 1 | Cargar los módulos (`%run`) - dispara todas sus pruebas internas |
| 2 | Ejecución del pipeline completo |
| 3 | Resultados (ranking IOI, información alfanumérica) |
| 4 | Verificación del mapa generado |

---
## 1. Cargar los módulos

Este notebook vive en la **raíz del proyecto**, al mismo nivel que `geokw_etl/`, `data/`, `output/` y `logs/`. El orden de los `%run` respeta las dependencias (p. ej. `transform_kpis.ipynb` usa funciones definidas en `extract_distritos.ipynb`).

In [30]:
%run geokw_etl/config.ipynb

Tabla maestra cargada: 11 distritos. poblacion total 697.233 hab.
Dataset de respaldo cargado: 31 estaciones


In [31]:
%run geokw_etl/logging_config.ipynb

2026-09-17 10:17:32 - INFO - Prueba de logging desde logging_config.ipynb


In [32]:
%run geokw_etl/extract_distritos.ipynb

'San Pablo - Santa Justa'        -> 'SAN PABLO-SANTA JUSTA'
'Nervión'                        -> 'NERVION'
'Bellavista - La Palmera'        -> 'BELLAVISTA-LA PALMERA'
'  Triana  '                     -> 'TRIANA'
        nombre_distrito      Area
  BELLAVISTA-LA PALMERA 16.035747
          CASCO ANTIGUO  4.227262
            CERRO-AMATE  7.499631
ESTE-ALCOSA-TORREBLANCA 31.379457
           LOS REMEDIOS 15.561799
               MACARENA  3.172876
                NERVION  3.202142
                  NORTE 38.566427
  SAN PABLO-SANTA JUSTA  5.625674
                    SUR  7.485120
                 TRIANA  9.325226


In [33]:
%run geokw_etl/extract_socioeconomico.ipynb

        nombre_distrito  poblacion_total  renta_media_neta_persona
           LOS REMEDIOS            25771                     21423
                NERVION            51276                     20984
          CASCO ANTIGUO            56980                     20777
                 TRIANA            47114                     17262
  BELLAVISTA-LA PALMERA            42647                     16488
  SAN PABLO-SANTA JUSTA            59018                     15838
                    SUR            69359                     15297
ESTE-ALCOSA-TORREBLANCA           107133                     13175
               MACARENA            76059                     12576
                  NORTE            70963                     12100
            CERRO-AMATE            90913                      9993


In [34]:
%run geokw_etl/extract_cargadores.ipynb

Tabla maestra cargada: 11 distritos. poblacion total 697.233 hab.
Dataset de respaldo cargado: 31 estaciones
2026-09-17 10:17:33 - INFO - Prueba de logging desde logging_config.ipynb



Origen: API
Estaciones disponibles: 533
Ejemplo: [{'nombre': 'Parking Albareda', 'lat': 37.38958069996167, 'lon': -5.995750113896634, 'potencia_kw': 7}, {'nombre': 'Parking Plaza Nueva', 'lat': 37.38906332169779, 'lon': -5.995629900618383, 'potencia_kw': 22}, {'nombre': 'Parking APK2 Magdalena', 'lat': 37.3907947, 'lon': -5.997415199999978, 'potencia_kw': 7.4}]
Estaciones dadas de alta en los ultimos 4 dias: 2
                                   nombre            fecha_creacion
                        McDonalds Sevilla 2026-09-15 08:06:00+00:00
Ayunt. Sevilla: Calle Baltasar de Alcázar 2026-09-15 08:04:00+00:00


In [35]:
%run geokw_etl/transform_geoprocessing.ipynb

Validos: 29 | Cuarentena: 2

En cuarentena (fuera de los 11 distritos):
                                  nombre                                       motivo_rechazo
         REE Cartuja (C/ Inca Garcilaso) Punto fuera de los 11 distritos oficiales de Sevilla
El Corte Ingles San Juan de Aznalfarache Punto fuera de los 11 distritos oficiales de Sevilla

Potencia agregada por distrito:
        nombre_distrito  potencia_total_kw  num_estaciones
          CASCO ANTIGUO              229.7              10
  SAN PABLO-SANTA JUSTA              194.0               5
                NERVION              119.7               5
                 TRIANA               88.0               4
                  NORTE               66.0               3
                    SUR               22.0               1
  BELLAVISTA-LA PALMERA               22.0               1
            CERRO-AMATE                0.0               0
               MACARENA                0.0               0
ESTE-ALCOSA-TORREBLANCA

In [36]:
%run geokw_etl/transform_kpis.ipynb

        nombre_distrito  poblacion_total  renta_media_neta_persona  potencia_total_kw  densidad_energetica  renta_norm  densidad_norm        IOI
           LOS REMEDIOS            25771                     21423                  0             0.000000    1.000000       0.000000 100.000000
                 TRIANA            47114                     17262                 22             4.669525    0.635958       0.250533  47.662970
  BELLAVISTA-LA PALMERA            42647                     16488                 22             5.158628    0.568241       0.276774  41.096675
                NERVION            51276                     20984                 66            12.871519    0.961592       0.690592  29.752426
                    SUR            69359                     15297                 66             9.515708    0.464042       0.510544  22.712828
ESTE-ALCOSA-TORREBLANCA           107133                     13175                 44             4.107045    0.278390       0.220

In [37]:
%run geokw_etl/visualize_mapa.ipynb

Mapa de prueba generado en ./output/mapa_sevilla.html


In [38]:
%run geokw_etl/pipeline.ipynb

`logging_config.ipynb` ya dejó el logging configurado, pero varias celdas de prueba de los módulos siguientes han escrito líneas de más en `logs/ejecucion_etl.log`. Se vuelve a llamar a `configurar_logging()` para dejarlo limpio antes de la ejecución real.

In [39]:
logger = configurar_logging()
print(f"Todos los modulos cargados. Logging reiniciado limpio -> {RUTA_LOG}")

Todos los modulos cargados. Logging reiniciado limpio -> ./logs/ejecucion_etl.log


---
## 2. Ejecución del pipeline completo

`ejecutar_pipeline()` (definida en `pipeline.ipynb`) encadena la extracción, el geoprocesamiento espacial, el cálculo de los tres KPIs y la generación del mapa. Observa en la salida los `INFO` de cada fase y, si aplica, los `WARNING` de la API de Open Charge Map.

In [40]:
master, df_validos_estaciones, df_cuarentena_estaciones, origen_estaciones = ejecutar_pipeline()

print(f"\nOrigen de los puntos de recarga: {origen_estaciones}")
print(f"Distritos procesados: {len(master)}")
print(f"Puntos de recarga validos (dentro de un distrito): {len(df_validos_estaciones)}")
print(f"Puntos en cuarentena (fuera de los 11 distritos): {len(df_cuarentena_estaciones)}")


Origen de los puntos de recarga: API
Distritos procesados: 11
Puntos de recarga validos (dentro de un distrito): 352
Puntos en cuarentena (fuera de los 11 distritos): 181


---
## 3. Resultados

Ranking de los 11 distritos por Índice de Oportunidad de Inversión - la información puramente alfanumérica que interesa revisar, independientemente del mapa.

In [41]:
cols_resultado = ["nombre_distrito", "poblacion_total", "renta_media_neta_persona",
                   "potencia_total_kw", "num_estaciones", "densidad_energetica",
                   "renta_norm", "densidad_norm", "IOI"]

tabla_resultado = master[cols_resultado].sort_values("IOI", ascending=False).reset_index(drop=True)
tabla_resultado.index = tabla_resultado.index + 1
print(tabla_resultado.round(2).to_string())

            nombre_distrito  poblacion_total  renta_media_neta_persona  potencia_total_kw  num_estaciones  densidad_energetica  renta_norm  densidad_norm    IOI
1             CASCO ANTIGUO            56980                     20777              314.0              22                55.11        0.94           0.00  94.35
2                   NERVION            51276                     20984             1391.7              33               271.41        0.96           0.53  44.78
3                       SUR            69359                     15297             1454.8              35               209.75        0.46           0.38  28.68
4     SAN PABLO-SANTA JUSTA            59018                     15838             1666.5              34               282.37        0.51           0.56  22.43
5   ESTE-ALCOSA-TORREBLANCA           107133                     13175             1677.4              37               156.57        0.28           0.25  20.86
6                  MACARENA       

In [42]:
if len(df_cuarentena_estaciones):
    print("Puntos de recarga en cuarentena (fuera de los 11 distritos):")
    print(df_cuarentena_estaciones[["nombre", "lat", "lon", "motivo_rechazo"]].to_string(index=False))
else:
    print("No hay puntos en cuarentena en esta ejecucion.")

Puntos de recarga en cuarentena (fuera de los 11 distritos):
                                                           nombre       lat       lon                                       motivo_rechazo
                                                E.S. Repsol Camas 37.381565 -6.026507 Punto fuera de los 11 distritos oficiales de Sevilla
                                                  Carrefour Camas 37.390999 -6.029740 Punto fuera de los 11 distritos oficiales de Sevilla
                                      CC San Juan de Aznalfarache 37.387120 -6.029818 Punto fuera de los 11 distritos oficiales de Sevilla
                                                  Carrefour Camas 37.392016 -6.030936 Punto fuera de los 11 distritos oficiales de Sevilla
                                                Ballenoil Tomares 37.378188 -6.029158 Punto fuera de los 11 distritos oficiales de Sevilla
                                       Mercadona Calle La Montaña 37.397003 -6.033730 Punto fuera de los 

---
## 4. Verificación del mapa generado

Comprobación de que `output/mapa_sevilla.html` existe y contiene los 5 elementos exigidos por la especificación de diseño UX/UI.

In [43]:
import os

assert os.path.exists(RUTA_SALIDA_MAPA), f"No se encontro el mapa en {RUTA_SALIDA_MAPA}"
tamano_kb = os.path.getsize(RUTA_SALIDA_MAPA) / 1024
print(f"Mapa generado: {RUTA_SALIDA_MAPA} ({tamano_kb:.0f} KB)")

with open(RUTA_SALIDA_MAPA, encoding="utf-8") as f:
    html_mapa = f.read()

comprobaciones = {
    "Tiles Esri WorldStreetMap": "arcgisonline.com/ArcGIS/rest/services/World_Street_Map" in html_mapa,
    "Escala grafica (control_scale)": "control.scale()" in html_mapa,
    "Flecha de Norte (color rojo)": "color:red" in html_mapa,
    "Leyenda estrecha (240 px)": "240" in html_mapa,
    "Clustering de puntos (MarkerCluster)": "markerClusterGroup" in html_mapa or "MarkerCluster" in html_mapa,
}
for nombre, ok in comprobaciones.items():
    print(f"  [{'OK' if ok else 'FALTA'}] {nombre}")

assert all(comprobaciones.values()), "Falta algun requisito de diseno UX/UI en el mapa generado"
print("\nLos 5 requisitos de diseno UX/UI estan presentes en el mapa.")

Mapa generado: ./output/mapa_sevilla.html (576 KB)
  [OK] Tiles Esri WorldStreetMap
  [OK] Escala grafica (control_scale)
  [OK] Flecha de Norte (color rojo)
  [OK] Leyenda estrecha (240 px)
  [OK] Clustering de puntos (MarkerCluster)

Los 5 requisitos de diseno UX/UI estan presentes en el mapa.


In [44]:
df_validos_export = df_validos_estaciones.rename(columns={"nombre_distrito": "distrito"}).assign(estado="Valido")
df_cuarentena_export = df_cuarentena_estaciones.assign(distrito="Fuera de distrito", estado="Cuarentena")

tabla_estaciones = pd.concat([df_validos_export, df_cuarentena_export], ignore_index=True)
tabla_estaciones = tabla_estaciones[["nombre", "distrito", "lat", "lon", "potencia_kw", "estado"]]
tabla_estaciones = tabla_estaciones.sort_values(["estado", "distrito", "nombre"]).reset_index(drop=True)

print(tabla_estaciones.to_string(index=False))
tabla_estaciones.to_csv("output/estaciones_carga.csv", index=False, encoding="utf-8")
print(f"\nGuardado: output/estaciones_carga.csv ({len(tabla_estaciones)} filas)")

                                                                 nombre                distrito       lat       lon  potencia_kw     estado
                                                     Mercadona Bolullos       Fuera de distrito 37.349172 -6.136712         22.0 Cuarentena
                                              ALDI Mairena del Aljarafe       Fuera de distrito 37.351530 -6.054560         11.0 Cuarentena
                                                           ALDI Tomares       Fuera de distrito 37.369111 -6.039324          7.4 Cuarentena
                                              Acciona - Parque Guadaíra       Fuera de distrito 37.353421 -5.858260        100.0 Cuarentena
                                      Aldi Alcalá de Guadaira (Atlante)       Fuera de distrito 37.324845 -5.853786        100.0 Cuarentena
                                                Aldi Bolullos (Atlante)       Fuera de distrito 37.350615 -6.138651        100.0 Cuarentena
                    